# 08 — Streaming：`stream_mode` 有好幾種「鏡頭」，對準的東西不一樣

**這份要學什麼**
- 五種 `stream_mode`（values / updates / messages / custom / debug）各自在拍什麼
- `get_stream_writer()`：node 裡自己回報「State 沒存、但想讓使用者看到」的進度

> 不需要 API key。`values` / `updates` / `debug` 模式跟模型無關，`messages` 模式用
> `scripted_model()`（照劇本念台詞的演員，見 `_llm.py`）模擬逐字吐字的效果。

`.invoke()` 是「看預錄好的完整影片」——整個圖跑完，一次把最終結果丟給你。`.stream()` 是
「看直播」——邊跑邊把「現在發生了什麼事」傳給你看，不用等到最後。

```
.invoke()                              .stream()
  │                                      │
  ▼                                      ▼
node1 執行中...                     node1 執行中... ──▶ 吐出一個 chunk
  │                                      │
  ▼                                      ▼
node2 執行中...                     node2 執行中... ──▶ 吐出一個 chunk
  │                                      │
  ▼                                      ▼
拿到「最終結果」                     全程都在收 chunk，最後一個就是最終狀態
（等全部做完才看到東西）              （邊做邊看，像看直播）
```

`stream_mode` 決定「直播鏡頭對準哪裡」——這份 notebook 會示範四種鏡頭角度：`values`
（對準每步做完的完整狀態）、`updates`（對準這步改了什麼）、`messages`（對準 LLM 正在
吐出的字）、`debug`（對準每個任務的詳細輸入輸出）。

In [ ]:
import sys

sys.path.insert(0, ".")

from langgraph.graph import END, START, MessagesState, StateGraph

from _graph_viz import show_graph


def step1(state: MessagesState) -> dict:
    return {"messages": [("ai", "第一步完成")]}


def step2(state: MessagesState) -> dict:
    return {"messages": [("ai", "第二步完成")]}


builder = StateGraph(MessagesState)
builder.add_node("step1", step1)
builder.add_node("step2", step2)
builder.add_edge(START, "step1")
builder.add_edge("step1", "step2")
builder.add_edge("step2", END)
graph = builder.compile()
show_graph(graph)

## `stream_mode="values"`：鏡頭對準「每步做完後，完整的 State 長怎樣」

每個 node 跑完，就吐出一次完整的 `messages` 清單。下面的輸出會看到 **3 個 chunk**，但圖只有
2 個 node（`step1`、`step2`）——因為第一個 chunk 是「輸入還沒被任何 node 處理過」的起始狀態，
之後每個 node 跑完再各吐一次。適合「我只想知道現在完整狀態長怎樣」，代價是資料量比較大
（每次都是全量）。

In [2]:
for chunk in graph.stream({"messages": [("human", "go")]}, stream_mode="values"):
    print([m.content for m in chunk["messages"]])

['go']
['go', '第一步完成']
['go', '第一步完成', '第二步完成']


## `stream_mode="updates"`：鏡頭只對準「這一步哪個 node 改了什麼」

不會給你完整狀態，只給「剛跑完的那個 node 回傳了什麼」，用 `{node 名稱: 回傳值}` 表示。
資料量小很多，適合做「進度指示器」（哪個 node 跑完了）或記錄每步的 diff。

In [3]:
for chunk in graph.stream({"messages": [("human", "go")]}, stream_mode="updates"):
    print(chunk)

{'step1': {'messages': [('ai', '第一步完成')]}}
{'step2': {'messages': [('ai', '第二步完成')]}}


## 一次開多種鏡頭：`stream_mode` 傳一個 list

想同時看「完整狀態」跟「哪步改了什麼」，就傳 `["values", "updates"]`。回傳的每個 chunk 會
多一個標籤（`mode`），告訴你這筆資料是哪種鏡頭拍的，方便同一個迴圈裡分流處理——下面輸出
可以看到 `values` 跟 `updates` 交錯出現。

In [4]:
for mode, chunk in graph.stream({"messages": [("human", "go")]}, stream_mode=["values", "updates"]):
    print(mode, "->", chunk)

values -> {'messages': [HumanMessage(content='go', additional_kwargs={}, response_metadata={}, id='efb8bca0-33ee-48d0-beaf-d28de8ecd91c')]}
updates -> {'step1': {'messages': [('ai', '第一步完成')]}}
values -> {'messages': [HumanMessage(content='go', additional_kwargs={}, response_metadata={}, id='efb8bca0-33ee-48d0-beaf-d28de8ecd91c'), AIMessage(content='第一步完成', additional_kwargs={}, response_metadata={}, id='6585976b-bc7e-4e07-a619-bd216740612e', tool_calls=[], invalid_tool_calls=[])]}
updates -> {'step2': {'messages': [('ai', '第二步完成')]}}
values -> {'messages': [HumanMessage(content='go', additional_kwargs={}, response_metadata={}, id='efb8bca0-33ee-48d0-beaf-d28de8ecd91c'), AIMessage(content='第一步完成', additional_kwargs={}, response_metadata={}, id='6585976b-bc7e-4e07-a619-bd216740612e', tool_calls=[], invalid_tool_calls=[]), AIMessage(content='第二步完成', additional_kwargs={}, response_metadata={}, id='a5a0b9be-3e15-46a2-bddf-5a0a65f438f0', tool_calls=[], invalid_tool_calls=[])]}


## `stream_mode="messages"`：鏡頭對準「LLM 正在吐出的字」

前面兩種鏡頭都是「node 執行完」才有東西；`messages` 模式是專門對準「模型正在生成的
token」——只有 node 裡面真的呼叫了 chat model，才會有輸出。這裡用 `scripted_model` 模擬。

拿到的是 `(chunk, metadata)` 一組一組的：`chunk.content` 是這個片段的文字，`metadata` 帶著
額外資訊。

In [5]:
from _llm import scripted_model

fake_model = scripted_model(["這是 一段 會被 逐字 串流 出來 的 示範 文字"])


def call_model(state: MessagesState) -> dict:
    return {"messages": [fake_model.invoke(state["messages"])]}


stream_builder = StateGraph(MessagesState)
stream_builder.add_node("call_model", call_model)
stream_builder.add_edge(START, "call_model")
stream_builder.add_edge("call_model", END)
stream_graph = stream_builder.compile()

for chunk, metadata in stream_graph.stream({"messages": [("human", "講一句話")]}, stream_mode="messages"):
    print(repr(chunk.content), "| from node:", metadata.get("langgraph_node"))

'這是' | from node: call_model
' ' | from node: call_model
'一段' | from node: call_model
' ' | from node: call_model
'會被' | from node: call_model
' ' | from node: call_model
'逐字' | from node: call_model
' ' | from node: call_model
'串流' | from node: call_model
' ' | from node: call_model
'出來' | from node: call_model
' ' | from node: call_model
'的' | from node: call_model
' ' | from node: call_model
'示範' | from node: call_model
' ' | from node: call_model
'文字' | from node: call_model


`GenericFakeChatModel`（`scripted_model` 底層用的類別）的 `_stream()` 會把文字依空白拆成
多個 chunk，模擬逐字輸出——所以上面才會看到好幾個小 chunk，而不是一次吐出整段文字。
如果接真的、支援 streaming 的 provider（例如 OpenAI），行為是同一套：一樣是 `stream_mode=
"messages"` 收到一連串 token chunk，只是真正的模型會依實際 tokenizer 切、而不是依空白切。
metadata 裡的 `langgraph_node` 告訴你這個 token 是哪個節點產生的，多 agent 場景（`09`）
很好用。

## 除錯用的鏡頭：`stream_mode="debug"`

對準「每個任務（task）的詳細執行資訊」，包含輸入輸出。適合開發階段追蹤某個 node 到底收到
什麼、吐出什麼；正式環境不建議常態打開（資料量很大）。

In [6]:
for chunk in graph.stream({"messages": [("human", "go")]}, stream_mode="debug"):
    print(chunk["type"], "->", chunk["payload"].get("name"))

task -> step1
task_result -> step1
task -> step2
task_result -> step2


## 自己架一台鏡頭：`stream_mode="custom"`

前面四種鏡頭拍的都是「State 變化」或「LLM 輸出」，規格是固定的。如果 node 裡面有你想
自己回報的東西——例如「爬蟲爬到第 3 頁了」「正在呼叫外部 API，稍等」——標準鏡頭拍不到，
因為那些進度訊息根本不會寫進 State。`get_stream_writer()` 讓你在 node 裡直接架一台
自己的鏡頭：想拍什麼就 `writer(...)`，配 `stream_mode="custom"` 收看，完全不影響
State 的內容。

In [7]:
from langgraph.config import get_stream_writer


def slow_task(state: MessagesState) -> dict:
    writer = get_stream_writer()
    writer({"progress": "開始處理"})
    writer({"progress": "處理到一半"})
    writer({"progress": "完成"})
    return {"messages": [("ai", "任務完成")]}


custom_builder = StateGraph(MessagesState)
custom_builder.add_node("slow_task", slow_task)
custom_builder.add_edge(START, "slow_task")
custom_builder.add_edge("slow_task", END)
custom_graph = custom_builder.compile()

for chunk in custom_graph.stream({"messages": [("human", "開始吧")]}, stream_mode="custom"):
    print(chunk)

{'progress': '開始處理'}
{'progress': '處理到一半'}
{'progress': '完成'}


## 小結：五種鏡頭，對準不同東西

| stream_mode | 鏡頭對準什麼 | 適合 |
|---|---|---|
| `values` | 每步後的完整 State | 想要每步都看到全貌 |
| `updates` | 每步哪個 node 改了什麼 | 輕量進度提示 / log |
| `messages` | 模型 token-level 輸出 + 來源 node | 前端邊生成邊顯示文字 |
| `custom` | 你自己 `writer(...)` 出來的任意內容 | State 裡沒有、但想回報的進度訊息 |
| `debug` | 每個任務的詳細輸入輸出 | 開發期除錯 |

`.invoke()` 永遠是「看預錄影片」；上面五種都是「看直播」的不同運鏡方式，依你要顯示給
使用者看的東西挑合適的模式。

下一份：`09_langgraph_multi_agent.ipynb`，用 supervisor pattern 跟 subgraph 組出多個
agent 協作。